# 02 – Preprocessing
**Owner:** Tony  
**Goal:** Clean the data, engineer home-minus-away features, and export the processed dataset used for modeling.  
**Study period:** 1999–2000 through 2024–25 (season-ending years 2000–2025) 
**Source:** `data/raw/public.dim_games.csv`.


This notebook preserves all valid regular-season games, documents structural missingness, and exports a processed dataset. **Important:** same-game box-score statistics are known only after the game; models using them classify completed games, not predict games before tipoff.

## Setup and load raw data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

source = RAW / 'public.dim_games.csv'
games = pd.read_csv(source, low_memory=False)
print('Raw data:', games.shape)
print('Columns:', len(games.columns))
display(games[['game_id','game_date','season','season_type','home_pts','away_pts']].head())

Raw data: (74569, 101)
Columns: 101


,game_id,game_date,season,season_type,home_pts,away_pts
0,198211270KCK,1982-11-27,1983,Regular Season,105,99
1,194611230WAS,1946-11-23,1947,Regular Season,74,50
2,198211300GOS,1982-11-30,1983,Regular Season,110,98
3,198212120POR,1982-12-12,1983,Regular Season,99,94
4,198212250UTH,1982-12-25,1983,Regular Season,101,111


## Filter study period and regular-season games

In [2]:
study = games.loc[games['season'].between(2000, 2025) & games['season_type'].eq('Regular Season')].copy()
print('Selected games:', len(study))
print('Number of seasons:', study['season'].nunique())
display(study.groupby('season').size().rename('games').to_frame())

Selected games: 31230
Number of seasons: 26


,games
season,
2000,1189
2001,1189
2002,1203
2003,1189
2004,1192
2005,1230
2006,1230
2007,1230
2008,1230


## Remove duplicates and missing data
Keep one row per `game_id`. Missing, negative, or tied final scores cannot define a reliable binary winner label. Count exclusions explicitly.

In [3]:
print('Duplicate game IDs:', study['game_id'].duplicated().sum())
study = study.drop_duplicates(subset='game_id', keep='first').copy()
for c in ['home_pts','away_pts']:
    study[c] = pd.to_numeric(study[c], errors='coerce')
invalid = (
    study[['home_pts','away_pts']].isna().any(axis=1)
    | study['home_pts'].lt(0)
    | study['away_pts'].lt(0)
    | study['home_pts'].eq(study['away_pts'])
)
print('Invalid-score rows removed:', invalid.sum())
study = study.loc[~invalid].copy()
print('Valid games:', len(study))

Duplicate game IDs: 0
Invalid-score rows removed: 0
Valid games: 31230


## Inspect missing data before deciding how to handle it
A missing value is not necessarily an invalid game. In this dataset, many team statistics are entirely absent for seasons 2016–2025. **Do not fill these missing statistics with zero** (zero would mean a team recorded no rebounds/assists). **Do not drop all such games** if the goal is to retain 26 seasons.

In [4]:
stats = ['fg_pct','ft_pct','fg3_pct','ast','reb','tov']
stat_columns = [f'{side}_{s}' for s in stats for side in ['home','away']]
for c in stat_columns:
    study[c] = pd.to_numeric(study[c], errors='coerce')
missing = study[stat_columns].isna().sum().sort_values(ascending=False)
display(pd.DataFrame({'missing_count':missing, 'missing_pct':(missing / len(study) * 100).round(1)}))
missing_by_season = study.groupby('season')[stat_columns].apply(lambda x: x.isna().mean().mul(100))
display(missing_by_season.round(1))

,missing_count,missing_pct
home_tov,14414,46.2
away_tov,14414,46.2
home_fg3_pct,13197,42.3
away_fg3_pct,13197,42.3
home_ast,13197,42.3
away_ast,13197,42.3
home_reb,13197,42.3
away_reb,13197,42.3
away_ft_pct,1590,5.1
away_fg_pct,1589,5.1


,home_fg_pct,away_fg_pct,home_ft_pct,away_ft_pct,home_fg3_pct,away_fg3_pct,home_ast,away_ast,home_reb,away_reb,home_tov,away_tov
season,,,,,,,,,,,,
2000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2002,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2
2003,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
2004,0.3,0.3,0.3,0.3,0.3,0.3,0.3,0.3,0.3,0.3,0.3,0.3
2005,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2006,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2007,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Check out-of-range values and possible outliers
Basketball box-score outliers may be legitimate. Report them rather than automatically removing them.

In [5]:
pct_columns = [f'{side}_{s}' for s in ['fg_pct','ft_pct','fg3_pct'] for side in ['home','away']]
for c in pct_columns:
    print(c, 'outside [0,1]:', int((study[c].notna() & ~study[c].between(0,1)).sum()))
for c in ['home_reb','away_reb','home_ast','away_ast','home_tov','away_tov']:
    print(c, 'negative:', int(study[c].lt(0).sum()))

home_fg_pct outside [0,1]: 0
away_fg_pct outside [0,1]: 0
home_ft_pct outside [0,1]: 0
away_ft_pct outside [0,1]: 0
home_fg3_pct outside [0,1]: 0
away_fg3_pct outside [0,1]: 0
home_reb negative: 0
away_reb negative: 0
home_ast negative: 0
away_ast negative: 0
home_tov negative: 0
away_tov negative: 0


## Feature engineering: home minus away differences
`HOME_TEAM_WINS` is 1 if the home team won, otherwise 0. Difference features are calculated only where both teams have values. Missing values remain `NaN` for a later modeling pipeline to handle explicitly.

In [6]:
study['HOME_TEAM_WINS'] = (study['home_pts'] > study['away_pts']).astype('int8')
feature_cols = []
for s in stats:
    feature = f'{s.upper()}_diff'
    study[feature] = study[f'home_{s}'] - study[f'away_{s}']
    feature_cols.append(feature)

print('Home win rate:', round(study['HOME_TEAM_WINS'].mean() * 100, 2), '%')
display(study[feature_cols + ['HOME_TEAM_WINS']].head())

Home win rate: 60.8 %


,FG_PCT_diff,FT_PCT_diff,FG3_PCT_diff,AST_diff,REB_diff,TOV_diff,HOME_TEAM_WINS
382,0.029,0.051,-0.032,8.0,-8.0,-8.0,1
536,0.069,0.000,NaN,NaN,NaN,NaN,1
538,NaN,NaN,NaN,NaN,NaN,NaN,1
550,NaN,NaN,NaN,NaN,NaN,NaN,0
561,0.120,0.000,NaN,NaN,NaN,NaN,0


## Choose final columns and verify
Do **not** include `home_pts`, `away_pts`, `home_plus_minus`, `away_plus_minus`, `home_wl`, or `away_wl` as model features: they reveal the game result. Game identifiers, dates, seasons and team names are retained as metadata, not automatically model inputs.

In [7]:
metadata = ['game_id','game_date','season','home_team_name','away_team_name']
processed = study[metadata + feature_cols + ['HOME_TEAM_WINS']].copy()
assert processed['game_id'].is_unique
assert processed['season'].between(2000, 2025).all()
assert set(processed['HOME_TEAM_WINS'].unique()).issubset({0,1})
print('Processed shape:', processed.shape)
display(processed.isna().sum().rename('missing').to_frame())
display(processed.head())

Processed shape: (31230, 12)


,missing
game_id,0
game_date,0
season,0
home_team_name,0
away_team_name,0
FG_PCT_diff,1777
FT_PCT_diff,1778
FG3_PCT_diff,13197
AST_diff,13197
REB_diff,13197


,game_id,game_date,season,home_team_name,away_team_name,FG_PCT_diff,FT_PCT_diff,FG3_PCT_diff,AST_diff,REB_diff,TOV_diff,HOME_TEAM_WINS
382,201202200GSW,2012-02-20,2012,Golden State Warriors,Los Angeles Clippers,0.029,0.051,-0.032,8.0,-8.0,-8.0,1
536,202312230NYK,2023-12-23,2024,New York Knicks,Milwaukee Bucks,0.069,0.000,NaN,NaN,NaN,NaN,1
538,200204080SEA,2002-04-08,2002,Seattle SuperSonics,LA Clippers,NaN,NaN,NaN,NaN,NaN,NaN,1
550,200211020WAS,2002-11-02,2003,Washington Wizards,New Jersey Nets,NaN,NaN,NaN,NaN,NaN,NaN,0
561,202312020MIA,2023-12-02,2024,Miami Heat,Indiana Pacers,0.120,0.000,NaN,NaN,NaN,NaN,0


## Export processed dataset

In [8]:
output = PROCESSED / 'games_processed.csv'
processed.to_csv(output, index=False)
print('Saved:', output.resolve())
print('Rows:', len(processed), '| Columns:', len(processed.columns))

Saved: /Users/tonynguyen/Desktop/csc3780-nba-win-prediction/data/processed/games_processed.csv
Rows: 31230 | Columns: 12


## Data limitations and handoff to modeling

- All 26 seasons are retained, but several difference features are missing in 2016–2025. A downstream model **cannot directly use these columns without a missing-data strategy**. Prefer obtaining a more complete source, or evaluate a restricted complete-statistics era separately. Median imputation alone would invent most of the missing values for ten consecutive seasons and can distort results.
- The `FG_PCT_diff` and `FT_PCT_diff` columns are more complete but still contain missing values.
- **Leakage / prediction timing:** `FG_PCT_diff`, `AST_diff`, etc. are *final game statistics*. They are useful for explaining/classifying outcomes **after the game**. For genuine *pre-game* predictions, replace these with each team's prior-game rolling averages, computed chronologically and shifted so the current game's results are excluded.
- If the group uses this file for modeling, perform imputation and train/test splitting inside the modeling pipeline; never fit an imputer using test data.
- Use a chronological train/test split when evaluating prediction of future seasons.

### Deliverables
`notebooks/02_preprocessing.ipynb` and `data/processed/games_processed.csv`.
